# Gradient Accumulation Notebook

> Hands-on Build It and Exercises.

## Build It

`code/main.py` is the runnable artifact. It does three things.

### Step 1: equivalence check

`equivalence_check()` builds two copies of the same network with the same seed. One sees a 16-sample batch in one forward pass. The other sees four 4-sample chunks with the loss divided by four. The function compares the gradient buffers before the optimizer step and the parameters after. The assertion is `max_abs_diff < 1e-4`.

### Step 2: sync-on-last-step pattern

`train_one_optimizer_step` walks micro-batches. For every micro-batch except the last it enters `no_sync_context(model)`. On a single process the context is a no-op; on DDP this is where the gradient all-reduce is skipped. The bookkeeping is the same regardless. A `sync_counter` records how many times we left the no_sync scope; for N micro-batches the count is one per effective step, not N.

### Step 3: the throughput curve

`sweep_effective_batches` runs the same model with a fixed micro-batch and a list of accumulation steps. For each setting it logs:

- `samples_per_sec`: total samples seen divided by wall time

- `median_step_ms`: 50th percentile per effective step

- `sync_calls`: collective points exercised

- `avg_loss`: average across the sweep's optimizer steps

The output lands in `outputs/accum-curve.json` and is reusable from a notebook.

Run it:

In [ ]:
```bash

python3 code/main.py

In [ ]:
```

The script prints the equivalence diff, then the sweep table, then the JSON path. Exit code zero.

## Exercises

In [ ]:
1. Re-run the sweep with `--num-steps 100` and plot samples per second against effective batch. Where does the curve flatten?
2. Add a wrong scaling variant (no division) and show the parameter diff at step 1 against the reference.
3. Swap SGD for AdamW and confirm the optimizer state advances once per effective step, not once per micro-batch.
4. Introduce a real `DistributedDataParallel` wrapper and route the `no_sync_context` to its method. Confirm sync_calls drops by N-1 per effective batch.
5. Modify the equivalence check to compare two different micro splits (2 by 8 vs 4 by 4) and explain any tolerance you need to relax.